In [1]:
import time

In [2]:
start_notebook = time.time()

In [3]:
import warnings
warnings.filterwarnings("ignore")

# 1. Parameters

In [4]:
name_dataset = 'DB_Pedia'
name_model = 'gpt-4o'
mode = 'zero'
seed = 3
part = 4

In [5]:
path_open = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post11/df_test_{part}.csv'

In [6]:
path_save = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post12/df_test_{part}.csv'

In [7]:
path_credentials = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/credentials/OPENAI_API_KEY.json'

# 2. Load Environment

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import json
import requests
import pandas as pd
from openai import OpenAI

In [10]:
with open(path_credentials, "r") as f:
    credentials = json.load(f)

In [11]:
OPENAI_API_KEY = credentials["OPENAI_API_KEY"]

In [12]:
client = OpenAI(api_key=OPENAI_API_KEY)

# 3. Functions

In [13]:
def zero_shot_prompt(text):

    intro = (
        "Classify the topic of the following text from the DBpedia Ontology dataset.\n"
        "Respond ONLY with a single digit (0–13) according to the category:\n"
        "0 = Company\n"
        "1 = EducationalInstitution\n"
        "2 = Artist\n"
        "3 = Athlete\n"
        "4 = OfficeHolder\n"
        "5 = MeanOfTransportation\n"
        "6 = Building\n"
        "7 = NaturalPlace\n"
        "8 = Village\n"
        "9 = Animal\n"
        "10 = Plant\n"
        "11 = Album\n"
        "12 = Film\n"
        "13 = WrittenWork\n"
        "Do not include any explanation or text, only the digit."
    )

    target = f'Text: "{text}"\nLabel:'
    return intro + target

In [14]:
def predict_label(text, prompt):

    try:

        start_time = time.perf_counter()
        first_token_time = None
        output_text = ""

        stream = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            top_p=1.0,
            seed=seed,
            stream=True,
            stream_options={"include_usage": True}
        )

        for chunk in stream:
            if first_token_time is None and len(chunk.choices) > 0 and chunk.choices[0].delta.content:
                first_token_time = time.perf_counter()

            if len(chunk.choices) > 0 and chunk.choices[0].delta.content:
                output_text += chunk.choices[0].delta.content

            if chunk.usage is not None:
                usage = chunk.usage

        end_time = time.perf_counter()

        return {
            "prediction": output_text.strip(),
            "latency_ms": (end_time - start_time) * 1000,
            "ttft_ms": (
                (first_token_time - start_time) * 1000
                if first_token_time else None
            ),
            "input_tokens": usage.prompt_tokens,
            "output_tokens": usage.completion_tokens
        }

    except:

        return {
            "prediction": '-1',
            "latency_ms": '-',
            "ttft_ms": '-',
            "input_tokens": '-',
            "output_tokens": '-'
        }

# 4. Load Dataset

In [15]:
df = pd.read_csv(path_open)

In [16]:
df.shape

(1400, 31)

In [17]:
pred_label = []
pred_latency = []
pred_ttft = []
pred_input_tokens = []
pred_output_tokens = []

In [18]:
for i in range(len(df)):

  text = df['text'].iloc[i]
  prompt = zero_shot_prompt(text)
  output = predict_label(text, prompt)

  pred_label.append(int(output['prediction']))
  pred_latency.append(output['latency_ms'])
  pred_ttft.append(output['ttft_ms'])
  pred_input_tokens.append(output['input_tokens'])
  pred_output_tokens.append(output['output_tokens'])

  if (i % 10) == 0:
    print(i)

0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340
350
360
370
380
390
400
410
420
430
440
450
460
470
480
490
500
510
520
530
540
550
560
570
580
590
600
610
620
630
640
650
660
670
680
690
700
710
720
730
740
750
760
770
780
790
800
810
820
830
840
850
860
870
880
890
900
910
920
930
940
950
960
970
980
990
1000
1010
1020
1030
1040
1050
1060
1070
1080
1090
1100
1110
1120
1130
1140
1150
1160
1170
1180
1190
1200
1210
1220
1230
1240
1250
1260
1270
1280
1290
1300
1310
1320
1330
1340
1350
1360
1370
1380
1390


In [19]:
df[f'{name_model}-{mode}-seed-{seed}-label'] = pred_label
df[f'{name_model}-{mode}-seed-{seed}-latency'] = pred_latency
df[f'{name_model}-{mode}-seed-{seed}-ttft'] = pred_ttft
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'] = pred_input_tokens
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'] = pred_output_tokens

In [20]:
df[f'{name_model}-{mode}-seed-{seed}-label'].value_counts()

,count
gpt-4o-zero-seed-3-label,
13,128
0,117
9,108
12,107
4,107
10,106
8,101
7,100
5,98


In [21]:
df[f'{name_model}-{mode}-seed-{seed}-latency'].describe()

,gpt-4o-zero-seed-3-latency
count,1400.000000
mean,447.752519
std,146.151892
min,295.543857
25%,378.060490
50%,420.381191
75%,471.624039
max,2846.824204


In [22]:
df[f'{name_model}-{mode}-seed-{seed}-ttft'].describe()

,gpt-4o-zero-seed-3-ttft
count,1400.000000
mean,445.474270
std,146.218274
min,294.683807
25%,375.880552
50%,418.923021
75%,469.086268
max,2840.952535


In [23]:
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'].describe()

,gpt-4o-zero-seed-3-input-tokens
count,1400.000000
mean,181.702857
std,35.270672
min,124.000000
25%,156.000000
50%,180.000000
75%,206.000000
max,682.000000


In [24]:
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'].describe()

,gpt-4o-zero-seed-3-output-tokens
count,1400.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


# 5. Save Dataset

In [25]:
df.to_csv(path_save, index = False)

# 6. Execution Time

In [26]:
end_notebook = time.time()

In [27]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 0h 10m 29.67s
